# Processing Nortek Signature1000 data for near-surface velocity

This notebook outlines the processing steps used to process Nortek Signature 1000 raw data files for the TIDE Guidance Note GN

## Step 1a

Perform vertical bin-mapping and side-lobe rejection using the Nortek SignatureViewer software. See the [Nortek SignatureViewer manual](https://www.nortekgroup.com/support/manuals/signature-series/signatureviewer-manual) for details. Export the velocity, amplitude, correlation and sensor data to .csv files.

## Step 1b

Read in the binary ".ad2cp" file using the [dolfyn python package](https://dolfyn.readthedocs.io/en/latest/). This package can read the raw binary files into an xarray dataset and has several functions for processing and analysing the data.

In [ ]:
import xarray as xr
import numpy as np # used to perform math operations
import pandas as pd # used to read csv files
import dolfyn # used to read adcp data
from dolfyn.adp import api

In [ ]:
ds = dolfyn.io.nortek2.read_signature("path/to/your/file.ad2cp")

Our instrument was configured to output data in "averaging" mode, which messes with dolfyns naming convention slightly (which typically uses burst mode). In practice, this means the binary reader appends "_avg" to some files. We need to remove this to avoid this to fit with the convention used by the rest of the package. Check your configuration and adjust/remove the code below if necessary.

In [ ]:
for key in list(ds.keys()):
    if 'time_avg' in ds[key].dims:
        ds[key] =  ds[key].rename({'time_avg':'time'})
ds = ds.drop(['time_avg'])
ds = ds.rename({'vel_avg':'vel','corr_avg':'corr','amp_avg':'amp','range_avg':'range','pressure_avg':'pressure','temp_avg':'temp','heading_avg':'heading','pitch_avg':'pitch','roll_avg':'roll','mag_avg':'mag',
'batt_avg':'batt','accel_avg':'accel','temp_clock_avg':'temp_clock','error_avg':'error','status_avg':'status','ensemble_avg':'ensemble',
'angrt_avg':'angrt','quaternions_avg':'quaternions','c_sound_avg':'c_sound','temp_press_avg':'temp_press','xmit_energy_avg':'xmit_energy'})

ds.attrs['cell_size'] = ds.attrs['cell_size_avg']
ds.attrs['rotate_vars'] = ['accel', 'angrt', 'mag', 'vel']

Now we can get our vetically bin-mapped and side-lobe rejected data into python and overwrite the revelant variables in the xarray dataset.

In [ ]:
ds_processed = xr.Dataset()
path_sens = r"path/to/your/sensor/csv/file.sen"
names = ['Month','Day','Year','Hour','Minute','Second','Error code','Status code','Battery voltage','Soundspeed','Heading','Pitch','Roll','Pressure','Temperature','Analog 1','Analog 2']
df_sens = pd.read_csv(path_sens, sep='\s+', header=None,names=names) 
df_sens['time'] = pd.to_datetime(df_sens[['Year','Month','Day','Hour','Minute','Second']],yearfirst=True)

for variable,unit in zip(['Heading','Pitch','Roll','Pressure','Temperature'],['degrees','degrees','degrees','dbar','degrees C']):
    ds_processed[variable] = xr.DataArray(df_sens[variable], dims=['time'], coords={'time':df_sens['time']})

# Read these values from your configuration file
blanking_dist = 0.1 # m
ncells = 29
cell_size = 1 # m
#

cell_centers = [blanking_dist + i*cell_size for i in range(1,ncells+1)]

paths = [r"path/to/signature/viewer/export/folder"+ext for ext in ['.a1','.a2','.a3','.a4']]
beam_amps = [pd.read_csv(path, sep='\s+', header=None).values for path in paths]
ds_processed['Amplitude'] = xr.DataArray(np.stack(beam_amps,axis=1), dims=['time','beam','distance'], coords={'time':df_sens['time'],'beam':[1,2,3,4],'distance':cell_centers})

paths = [r"path/to/signature/viewer/export/folder"+ext for ext in ['.c1','.c2','.c3','.c4']]
beam_corrs = [pd.read_csv(path, sep='\s+', header=None).values for path in paths]
ds_processed['Correlation'] = xr.DataArray(np.stack(beam_corrs,axis=1), dims=['time','beam','distance'], coords={'time':df_sens['time'],'beam':[1,2,3,4],'distance':cell_centers})

paths = [r"path/to/signature/viewer/export/folder"+ext for ext in ['.v1','.v2','.v3','.v4']]
beam_vels = [pd.read_csv(path, sep='\s+', header=None).values for path in paths]
ds_processed['Velocity'] = xr.DataArray(np.stack(beam_vels,axis=1), dims=['time','beam','distance'], coords={'time':df_sens['time'],'beam':[1,2,3,4],'distance':cell_centers})

# replace raw data with the SignatureViewer processed data (note, I had to swap axes to get the dimensions to match up)
ds['vel'] =  xr.DataArray(np.swapaxes(np.swapaxes(ds_processed['Velocity'].values,0,1),1,2), dims=['dir','range','time'], attrs= ds['vel'].attrs)
ds['amp'] =  xr.DataArray(np.swapaxes(np.swapaxes(ds_processed['Amplitude'].values,0,1),1,2), dims=['dir','range','time'], attrs= ds['amp'].attrs)
ds['corr'] =  xr.DataArray(np.swapaxes(np.swapaxes(ds_processed['Correlation'].values,0,1),1,2), dims=['dir','range','time'], attrs= ds['corr'].attrs)

You can use the dolfyn package to determine the depth of the instrument as a function of time and remove data aboce the surface. See the [dolfyn documentation](https://dolfyn.readthedocs.io/en/latest/) for details.

In [ ]:
api.clean.find_surface_from_P(ds,salinity=35)
ds = api.clean.nan_beyond_surface(ds)

## Step 2
Remove data where the correlation is below 50%.

In [ ]:
ds = ds.where(ds['corr'].max('dir') > 50)

## Step 3
Remove the bin nearest to the ADCP transducer to avoid frame interference.

In [ ]:
# remove first index in range dimension
ds = ds.isel(range=slice(1,len(ds.range+1)))

# Step 4
Convert the data from distance from the instrument to depth below the surface.

In [ ]:
# make new variable z_depth which is depth of each bin as a function of time in meters below the surface (negative)
# this can change depending on the datum you have measured from (i.e. distance from instrument or distance from the sea bed)
ds['z_depth'] = (ds['range'] - ds.attrs['h_deploy']) - ds['pressure']

# crop out all data when z_depth is positive (above the surface)
ds = ds.where(ds['z_depth'] < 0)

# get u and v velocity components
ds['u'] = ds['vel'].sel(dir='E')
ds['v'] = ds['vel'].sel(dir='N')

# lets interpolate the data onto 1 m bins in meters below the surface and put in a new xarray dataset

z_bins = np.arange(-20, 0, 1)

ds_z_depth = xr.Dataset()

# initialise u and v arrays
u_z_depth = np.zeros((len(ds.time), len(z_bins)))
v_z_depth = np.zeros((len(ds.time), len(z_bins)))

# loop through time index and interpolate onto fixed depths below the surface
for t,time in enumerate(ds.time.values):
    u_z_depth[t,:] = np.interp(z_bins, ds['z_depth'].sel(time=time).values, ds['u'].sel(time=time).values)
    v_z_depth[t,:] = np.interp(z_bins, ds['z_depth'].sel(time=time).values, ds['v'].sel(time=time).values)

ds_z_depth['u'] = xr.DataArray(u_z_depth, dims=['time', 'z_depth'], coords={'time': ds.time, 'z_depth': z_bins})
ds_z_depth['v'] = xr.DataArray(v_z_depth, dims=['time', 'z_depth'], coords={'time': ds.time, 'z_depth': z_bins})

# Step 5
Average the 3 second data to 1 minute average to remove the effects of the surface wave field

In [ ]:

ds_z_depth = ds_z_depth.resample(time='1min').mean()